In [2]:
import pandas as pd
import numpy as np
import os

In [ ]:
# Load CSV file
df = pd.read_csv("../data/g_data.csv", header=None)
output_dir = "g_data"
os.makedirs(output_dir, exist_ok=True)

# Process each row
for i, row in df.iterrows():
    header = row[2].strip(">").strip()
    sequence = row[3].strip()
    with open(f"{output_dir}/{header}.fasta", "w") as f:
        f.write(f">{header}\n{sequence}\n")

print("FASTA files saved!")

In [3]:
# Set path location for the pssm files generated and for the output csv file

pssm_dir = "PSSM_Features/gdata"
output_csv = "../features/pssm_features.csv"

# To read pssm files

def read_pssm(file_path):
    with open(file_path, "r") as f:
        lines = f.readlines()
    pssm_lines = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) >= 22 and parts[0].isdigit():
            try:
                pssm_row = list(map(int, parts[2:22]))
                pssm_lines.append(pssm_row)
            except:
                continue
    return np.array(pssm_lines)


# 1st Feature: Amino Acid Occurrence

def pssm_aao(matrix):
    return np.sum(matrix, axis=0) # column-wise sum


# 2nd Feature: Segment Distribution
def pssm_sd(matrix, fp=25):
    L, C = matrix.shape  # L: sequence length, C: number of amino acids (20)
    V = []
    for j in range(C):
        col = matrix[:, j]
        Tj = np.sum(col)  # Total substitution score for this column
        thresholds = [fp, 2*fp]  # Define segment percentage thresholds
        for t in thresholds:
            target = Tj * t / 100.0  # Forward cumulative target
            cum = 0
            for i in range(L):
                cum += col[i]
                if cum >= target:
                    V.append(i+1)
                    break
        for t in thresholds:
            target = Tj * t / 100.0  # Backward cumulative target
            cum = 0
            for i in reversed(range(L)):
                cum += col[i]
                if cum >= target:
                    V.append(L - i)
                    break
    return np.array(V)  # 4 features per column × 20 = 80-dimensional output


# 3rd Feature: Segmented Auto Covariance
def pssm_sac(matrix, df=5):  # 5 -> 100 features and 10 -> 200 features
    L, C = matrix.shape
    features = []
    for j in range(C):  # Iterate over each column (amino acid)
        for k in range(1, df+1):  # Lag values from 1 to df
            if L <= k:
                features.append(0.0)
                continue
            cov = np.mean(matrix[:L-k, j] * matrix[k:, j])  # Auto-covariance for lag k
            features.append(cov)
    return np.array(features)  # Output: 20 × df = 100-dimensional vector







In [ ]:
feature_rows = []
protein_ids = []
count = 0
for filename in os.listdir(pssm_dir):
    if filename.endswith(".pssm"):
        count = count + 1
        print(count)
        prot_id = filename.replace(".pssm", "") # Extract protein id from filename
        filepath = os.path.join(pssm_dir, filename)
        try:
            mat = read_pssm(filepath) # Load PSSM matric
            aao = pssm_aao(mat)
            sd = pssm_sd(mat)
            sac = pssm_sac(mat)
            full_vector = np.concatenate([aao, sd, sac]) # Combine all features
            feature_rows.append(full_vector)
            protein_ids.append(prot_id)

        except Exception as e: # Handle bad or corrupt files
            print(f"Error processing {filename}: {e}")

In [5]:
# --- SAVE OUTPUT ---
columns = [f"F{i+1}" for i in range(200)] # Generate column names F1 to F200
features_df = pd.DataFrame(feature_rows, index=protein_ids, columns=columns)
features_df.index.name = "Protein_ID"
features_df.to_csv(output_csv)

print(f"Features saved to: {output_csv}")

Features saved to: pssm_features.csv


In [6]:
# 4th csv
def extract_autocovariance_feature(matrix, df=10):
    matrix = matrix[:, :3]  # only the first 3 columns of SPD3 (H/E/C)
    L, C = matrix.shape  # L: number of sequence length, C: number of structural features
    features = []  # store Vk,j value

    for k in range(1, df + 1):
        for j in range(C):  # j: column index (0 to 2 for H, E, C)
            sum_ = 0.0
            for i in range(L - k):
                sum_ += matrix[i, j] * matrix[i + k, j]
            features.append(sum_ / L)
    return np.array(features)  # Return a 30-dimensional feature vector

In [7]:
# Batch extract Auto Covariance features from SPD3 (.i1s) files and save to CSV

def batch_extract_autocovariance(input_dir, output_csv_path, df=10):
    feature_list = []     # Stores the AC features for each protein
    sequence_id = []      # Stores the protein IDs

    for filename in os.listdir(input_dir):
        if filename.endswith(".i1s"):  # SPD3 output file
            file_path = os.path.join(input_dir, filename)
            try:
                matrix = np.loadtxt(file_path)  # Load SPD3 matrix
                features = extract_autocovariance_feature(matrix, df)  # Extract features using lag df
                feature_list.append(features)
                sequence_id.append(os.path.splitext(filename)[0])  # Use filename (no extension) as ID
            except Exception as e:
                print(f"Error processing {filename}: {e}")

    # Convert to DataFrame and save as CSV
    df_result = pd.DataFrame(feature_list, index=sequence_id, columns=[f"AC_{j + 1}" for j in range(df * 3)])
    df_result.to_csv(output_csv_path)
    print(f"Features saved to {output_csv_path}")


#input_folder = "output"
input_folder = 'output'
output_csv = "../features/auto_cov_features.csv"
batch_extract_autocovariance(input_folder, output_csv)


Features saved to auto_cov_features.csv


In [8]:
# 5th feature
def extract_composition_feature(matrix):
    matrix = matrix[:, 3:7]  # Select torsion angles columns: columns 3 to 6
    L, C = matrix.shape  # L = number of residues (rows), C = number of torsion columns (should be 4)
    features = []

    # Sum each column, then divide by L, to implement the formula in Algorithm 4
    for j in range(C):
        sum_ = 0.0
        for i in range(L):
            sum_ += matrix[i, j]
        features.append(sum_ / L)
    return np.array(features)  # Return a 4-dimensional feature vector


In [9]:
# Batch process all .i1s files in input folder
def batch_extract_composition(input_dir, output_csv_path):
    feature_list = []
    sequence_id = []

    for filename in os.listdir(input_dir):
        if filename.endswith(".i1s"):
            file_path = os.path.join(input_dir, filename)
            try:
                matrix = np.loadtxt(file_path)
                features = extract_composition_feature(matrix)
                feature_list.append(features)
                sequence_id.append(os.path.splitext(filename)[0])
            except Exception as e:
                print(f"Error processing {filename}: {e}")


    df_result = pd.DataFrame(
        feature_list,
        index=sequence_id,
        columns=["Composition_phi", "Composition_psi", "Composition_theta", "Composition_tau"]
    )
    df_result.to_csv(output_csv_path)
    print(f"✔️ Composition features saved to: {output_csv_path}")


#input_folder = "output"
input_folder = 'output'
output_csv = "../features/composition_features.csv"
batch_extract_composition(input_folder, output_csv)

✔️ Composition features saved to: composition_features.csv


In [10]:
# 6th features
def extract_one_lead_bigram_asa(matrix):
    asa_column = matrix[:, 6]  # ASA is the 7th column (index 6)
    L = len(asa_column)  # Protein length
    sum_ = 0.0

    for i in range(L - 2):
        sum_ += asa_column[i] * asa_column[i + 2]

    feature = sum_ / L
    return np.array([feature])

In [12]:
# Batch processing function to extract features from all .i1s files in a folder
def batch_extract_one_lead_bigram_asa(input_dir, output_csv_path):
    feature_list = []  # Store each protein’s extracted feature vector
    sequence_id = []  # Store sequence ID

    for filename in os.listdir(input_dir):
        if filename.endswith(".i1s"):  # Only process SPD3 output files
            file_path = os.path.join(input_dir, filename)
            try:
                matrix = np.loadtxt(file_path)  # Load SPD3 matrix
                feature = extract_one_lead_bigram_asa(matrix)  # Extract feature
                feature_list.append(feature)
                sequence_id.append(os.path.splitext(filename)[0])  # Remove .i1s suffix
            except Exception as e:
                print(f"Error processing {filename}: {e}")

    # Convert the features to a DataFrame and save to CSV
    import pandas as pd
    df_result = pd.DataFrame(feature_list, index=sequence_id, columns=["OLB_ASA"])
    df_result.to_csv(output_csv_path)
    print(f"One-Lead Bi-gram ASA features saved to: {output_csv_path}")


input_folder = "output"
output_csv = "../features/one_lead_bigram_asa.csv"
batch_extract_one_lead_bigram_asa(input_folder, output_csv)


One-Lead Bi-gram ASA features saved to: one_lead_bigram_asa.csv
